# Descarga de ensamblados → Google DriveBaja los genomas de referencia desde NCBI y los escribe directo en`Mi unidad/tesis/70_genomas/`. No pasan por tu disco local.**Por qué acá y no en la sesión de Claude:** esa sesión tiene bloqueada lasalida a NCBI y Ensembl por política del proxy. Colab no.**Orden:** montar Drive → verificar candidatos → descargar → pegar elresultado de vuelta en `data/genomas.tsv`.Los 6 ensamblados de abajo están como `candidato`: sonpropuestas **sin comprobar**. La celda de verificación los contrasta contraNCBI antes de que se baje nada.

## 1. Montar Drive

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import os, hashlib, json, urllib.request, zipfile, io, pathlibDESTINO = pathlib.Path('/content/drive/MyDrive/tesis/70_genomas')DESTINO.mkdir(parents=True, exist_ok=True)print('destino:', DESTINO)print('existe:', DESTINO.exists())

## 2. Candidatos a verificarSnapshot de `data/genomas.tsv`. Si lo cambiaste en el repo, volvé a generar elnotebook con `./scripts/gen_colab_notebook.py`.

In [ ]:
CANDIDATOS = [  {    "org": "cloro",    "especie": "Clonostachys rosea",    "assembly": "?",    "accession": "",    "confianza": "nula"  },  {    "org": "prupe",    "especie": "Prunus persica",    "assembly": "Prunus_persica_NCBIv2",    "accession": "GCF_000346465.2",    "confianza": "alta"  },  {    "org": "maldo",    "especie": "Malus domestica",    "assembly": "ASM211411v1 (GDDH13 v1.1)",    "accession": "GCF_002114115.1",    "confianza": "media"  },  {    "org": "gadmo",    "especie": "Gadus morhua",    "assembly": "gadMor3.0",    "accession": "GCF_902167405.1",    "confianza": "alta"  },  {    "org": "galga",    "especie": "Gallus gallus",    "assembly": "bGalGal1.mat.broiler.GRCg7b",    "accession": "GCF_016699485.2",    "confianza": "alta"  },  {    "org": "maggi",    "especie": "Magallana gigas",    "assembly": "cgigas_uk_roslin_v1",    "accession": "GCF_902806645.1",    "confianza": "alta"  }]for c in CANDIDATOS:    print(f"{c['org']:8} {c['confianza']:6} {c['accession'] or '(sin candidato)':20} {c['assembly']}")

## 3. Verificar contra NCBIPara cada organismo pregunta dos cosas: si el accession propuesto existe, ycuál es el ensamblado de **referencia vigente** de la especie. La segundaimporta más que la primera — un accession puede existir y no ser el quecorresponde.

In [ ]:
import urllib.parseAPI = "https://api.ncbi.nlm.nih.gov/datasets/v2alpha"def get(url):    try:        with urllib.request.urlopen(url, timeout=60) as r:            return json.load(r)    except Exception as e:        return {"_error": str(e)}def verifica(c):    out = {"org": c["org"], "candidato": c["accession"], "coincide": None}    if c["accession"]:        d = get(f"{API}/genome/accession/{c['accession']}/dataset_report")        reps = d.get("reports") or []        if reps:            r = reps[0]            out["ncbi_nombre"] = r.get("assembly_info", {}).get("assembly_name")            out["ncbi_organismo"] = r.get("organism", {}).get("organism_name")        else:            out["ncbi_nombre"] = None            out["error"] = "accession no encontrado"    esp = urllib.parse.quote(c["especie"])    d = get(f"{API}/genome/taxon/{esp}/dataset_report"            "?filters.reference_only=true&page_size=3")    reps = d.get("reports") or []    if reps:        r = reps[0]        out["referencia_vigente"] = r.get("accession")        out["referencia_nombre"] = r.get("assembly_info", {}).get("assembly_name")        out["nivel"] = r.get("assembly_info", {}).get("assembly_level")        out["coincide"] = (out["referencia_vigente"] == c["accession"])    else:        out["referencia_vigente"] = None        out["nota"] = "sin ensamblado de referencia; buscar por cepa"    return outRESULTADOS = [verifica(c) for c in CANDIDATOS]for r in RESULTADOS:    marca = "OK " if r.get("coincide") else "REVISAR"    print(f"\n[{marca}] {r['org']}")    print(f"   candidato : {r.get('candidato') or '(ninguno)'}  {r.get('ncbi_nombre') or ''}")    print(f"   vigente   : {r.get('referencia_vigente')}  {r.get('referencia_nombre') or ''}"          f"  nivel={r.get('nivel')}")    if r.get("nota"):        print(f"   ! {r['nota']}")

## 4. Elegir qué bajar**Mirá la salida de arriba antes de correr esto.** Por defecto se baja elensamblado **de referencia vigente**, no el candidato — si difieren, gana NCBI.Para forzar otro accession, editá `A_BAJAR` a mano.

In [ ]:
A_BAJAR = {r["org"]: r.get("referencia_vigente")           for r in RESULTADOS if r.get("referencia_vigente")}# Editá acá si querés forzar alguno, por ejemplo:# A_BAJAR["cloro"] = "GCA_XXXXXXXXX.1"faltan = [r["org"] for r in RESULTADOS if not r.get("referencia_vigente")]if faltan:    print("SIN RESOLVER (hay que buscarlos por cepa):", faltan)print()for o, a in sorted(A_BAJAR.items()):    print(f"{o:8} -> {a}")

## 5. Descargar a DriveEscribe `<org>/<accession>.fna.gz` en `70_genomas/` y calcula el `sha256`.Si el archivo ya está, lo saltea, así que se puede re-ejecutar sin problema.

In [ ]:
def baja(org, acc):    destino = DESTINO / org    destino.mkdir(parents=True, exist_ok=True)    final = destino / f"{acc}.fna.gz"    if final.exists() and final.stat().st_size > 0:        print(f"== {org}: ya está ({final.stat().st_size/1e6:.0f} MB)")        return None    url = (f"{API}/genome/accession/{acc}/download"           "?include_annotation_type=GENOME_FASTA")    print(f"== {org}: bajando {acc} ...")    with urllib.request.urlopen(url, timeout=1800) as r:        blob = r.read()    import gzip    with zipfile.ZipFile(io.BytesIO(blob)) as z:        nombres = [n for n in z.namelist()                   if n.endswith("_genomic.fna") and f"/{acc}/" in n]        if not nombres:            print(f"   ERROR: no encontré el FASTA en el zip de {acc}")            print("  ", z.namelist()[:10])            return None        crudo = z.read(nombres[0])    # Se escribe comprimido: son cientos de MB y Drive cobra por byte.    tmp = final.with_suffix(".gz.parcial")    with gzip.open(tmp, "wb", compresslevel=6) as fh:        fh.write(crudo)    tmp.rename(final)    h = hashlib.sha256()    with final.open("rb") as fh:        for chunk in iter(lambda: fh.read(1 << 20), b""):            h.update(chunk)    sha = h.hexdigest()    (destino / f"{acc}.fna.gz.sha256").write_text(f"{sha}  {acc}.fna.gz\n")    print(f"   {final.stat().st_size/1e6:.0f} MB  sha256={sha[:16]}...")    return shaSHAS = {}for org, acc in sorted(A_BAJAR.items()):    s = baja(org, acc)    if s:        SHAS[org] = (acc, s)

## 6. Cerrar el círculo con gitImprime las líneas para pegar en `data/genomas.tsv` y en`data/genomas.sha256`. **Ese paso es el que deja constancia versionada de quégenoma se usó** — el checksum guardado solo al lado del FASTA en Drive noprueba nada, porque quien reemplace el genoma reemplaza el checksum con él.

In [ ]:
import datetimehoy = datetime.date.today().isoformat()print("--- pegar en data/genomas.sha256 ---")print("org\taccession\tassembly\tsha256\tfecha_utc")for org, (acc, sha) in sorted(SHAS.items()):    nombre = next((r.get("referencia_nombre") for r in RESULTADOS                   if r["org"] == org), "")    print(f"{org}\t{acc}\t{nombre}\t{sha}\t{hoy}")print()print("--- en data/genomas.tsv, poner estado=verificado en:", sorted(SHAS), "---")

## Lo que esto NO resuelveSirve para los genomas, que son unos pocos GB. **No sirve para los BAMs**: sondel orden de 340 GB y las sesiones de Colab son efímeras y con límite detiempo. Ese volumen sigue yendo por `rclone` desde tu máquina, con`./scripts/drive_push.sh bam <org> --go`.Tampoco sirve para el alineamiento: 30-40 h de bowtie no entran en una sesiónde Colab.